# 01. Variables & Memory Management (5+ Years Interview Guide)
Comprehensive revision of CPython memory architecture, reference counting, object binding, integer caching, string interning, and generational garbage collection.

### Key 5-Year Interview Concepts Covered:
- **Object Reference Model**: Variables are pointers to heap objects, not memory containers.
- **Identity (`is`) vs Equality (`==`)**: Pointer address matching vs value comparator protocol (`__eq__`).
- **CPython Small Integer & String Interning**: Pre-allocated singletons (-5 to 256) and compile-time string interning.
- **Garbage Collection Dual System**: Deterministic reference counting + cyclic generational GC (Gen 0, 1, 2).

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. Variable Definition & Reference Binding
**Explanation**: In Python, variables do not store values directly; they are named references (pointers) bound to objects created in heap memory. Assigning `a = 100` allocates a PyObject in memory and binds the identifier `a` to its memory address `id(a)` in the current namespace dictionary. Multiple variables can point to the exact same underlying object.

**Syntax**: `variable_name = object_value`

In [2]:
payout_amounts = [10.0, 20.0]
reference_amounts = payout_amounts
print('Identity match?:', payout_amounts is reference_amounts)

Identity match?: True


### 2. Multiple Variable Assignment
**Explanation**: Python evaluates the right-hand side expressions first, bundles them into an internal tuple, and then unpacks each reference sequentially into the left-hand targets. This provides a clean, atomic way to initialize multiple independent references in a single bytecode instruction.

**Syntax**: `var1, var2, var3 = val1, val2, val3`

In [3]:
transaction_id, customer_id = 'TX100', 'C909'
print('Tx ID:', transaction_id, '| Cust ID:', customer_id)

Tx ID: TX100 | Cust ID: C909


### 3. Chained Variable Assignment
**Explanation**: Chained assignment binds multiple variable names to the exact same object reference in memory. Unlike separate assignments, `a = b = c = []` makes all three variables point to the identical mutable list instance in heap memory—mutating one affects all. For immutables (like numbers/strings), reassigning one simply re-binds that variable without affecting others.

**Syntax**: `var1 = var2 = var3 = shared_value`

In [4]:
first_limit = second_limit = default_limit = 5000
print('Equal ids?:', id(first_limit) == id(second_limit) == id(default_limit))

Equal ids?: True


### 4. Variable Scope Namespaces (Local Scope)
**Explanation**: Variables defined inside a function belong to the local namespace `locals()`. CPython optimizes local variable access using `LOAD_FAST` bytecode instructions (indexing a fixed-size array in the stack frame rather than doing hash table lookups), making local variable lookups significantly faster than global lookups.

**Syntax**: `def func(): local_var = value`

In [5]:
def calculate_local_fee():
    standard_fee = 5.0
    return standard_fee
print('Fee:', calculate_local_fee())

Fee: 5.0


### 5. Variable Scope Namespaces (Global Scope)
**Explanation**: Module-level variables live in the global namespace dictionary `globals()`. When accessed inside a function without the `global` keyword, they can be read. However, attempting to reassign or modify a global variable inside a function without declaring `global var_name` raises an `UnboundLocalError` because Python marks any assigned variable as local at compile-time.

**Syntax**: `global global_variable_name`

In [6]:
global_payout_limit = 10000
print('Global access:', global_payout_limit)

Global access: 10000


### 6. Reference Counting Internals
**Explanation**: CPython's primary memory management mechanism is deterministic reference counting. Every `PyObject` header contains an `ob_refcnt` field tracking active pointers. When `sys.getrefcount(obj)` is called (which temporarily adds 1 for the argument itself), it returns the active references. When `ob_refcnt` drops to zero, the object's memory is deallocated immediately without waiting for a GC cycle.

**Syntax**: `import sys; sys.getrefcount(target_object)`

In [7]:
import sys
transaction_record_list = [100.0, 200.0]
print('Refs:', sys.getrefcount(transaction_record_list) - 1)

Refs: 1


### 7. Small Integer Caching (-5 to 256)
**Explanation**: CPython pre-allocates and caches an array of small integer objects in the range `[-5, 256]` during interpreter startup. Any variable assigned an integer in this range points to the shared singleton instance in memory, causing `a is b` to evaluate to `True`. Integers outside this range allocate distinct objects in heap memory (unless folded by the compiler in the same code object).

**Syntax**: `int_a is int_b  # True if within [-5, 256]`

In [8]:
cached_integer_one = 256
cached_integer_two = 256
print('256 pooled?:', cached_integer_one is cached_integer_two)
uncached_integer_one = 300
uncached_integer_two = 300
print('300 pooled?:', uncached_integer_one is uncached_integer_two)

256 pooled?: True
300 pooled?: True


### 8. String Interning
**Explanation**: CPython automatically interns string literals that look like valid Python identifiers (ASCII letters, digits, underscores) at compile time. Interned strings share the same memory location, allowing instantaneous pointer comparison `O(1)` using `is` rather than character-by-character string comparison `O(N)`. Developers can explicitly intern strings using `sys.intern(string)`.

**Syntax**: `import sys; sys.intern(string_value)`

In [9]:
import sys
interned_string_one = sys.intern('transaction_id_9901')
interned_string_two = sys.intern('transaction_id_9901')
print('Intern match:', interned_string_one is interned_string_two)

Intern match: True


### 9. Deleting Namespace Bindings (`del`)
**Explanation**: The `del` keyword does not delete objects directly from memory; it deletes the variable name binding from the current namespace dictionary and decrements the object's `ob_refcnt` by 1. If that reference count hits 0, CPython automatically frees the object from heap memory and invokes its `__del__` destructor if present.

**Syntax**: `del variable_name` / `del collection[key_or_index]`

In [10]:
temporary_data_list = [10, 20]
del temporary_data_list
try:
    print(temporary_data_list)
except NameError as error_message:
    print('Unbound safely:', error_message)

Unbound safely: name 'temporary_data_list' is not defined


### 10. Memory Address Identity (`is`) vs Value Equality (`==`)
**Explanation**: The `==` operator checks for value equality by invoking the object's `__eq__()` method (e.g. do these two lists contain identical elements?). The `is` operator checks for object identity by comparing raw memory addresses `id(a) == id(b)`. In production, always use `is` for singletons like `None`, `True`, `False`, and `==` for data value comparisons.

**Syntax**: `obj1 is obj2  # Pointer check` / `obj1 == obj2  # Value equality`

In [11]:
first_ledger_list = [10, 20]
second_ledger_list = [10, 20]
print('== match:', first_ledger_list == second_ledger_list, '| is match:', first_ledger_list is second_ledger_list)

== match: True | is match: False


### 11. Generational Garbage Collection (`gc`)
**Explanation**: While reference counting handles 95%+ of memory deallocation, it cannot free reference cycles (e.g., Object A points to Object B, and Object B points to Object A). CPython's cyclic garbage collector operates on three generations (Gen 0, 1, 2) using a doubly linked list to detect unreachable object cycles and reclaim memory. Developers can inspect thresholds and force collections via the `gc` module.

**Syntax**: `import gc; gc.collect()` / `gc.get_threshold()`

In [12]:
import gc
print('Collected cyclic dependencies:', gc.collect())

Collected cyclic dependencies: 11027


### 12. Shared References with Mutables (Aliasing Trap)
**Explanation**: When two variables reference the same mutable object (such as a list, dictionary, or set), modifying the object via one variable mutates the shared underlying heap memory and is reflected across all referencing variables. To prevent unintended side effects, create a shallow copy via `.copy()` or `[:]`, or a deep copy via `copy.deepcopy()`.

**Syntax**: `list_b = list_a.copy()  # Prevents top-level aliasing`

In [13]:
original_ledger_list = []
sharing_ledger_list = original_ledger_list
original_ledger_list.append(99.99)
print('Sharing list reflects changes:', sharing_ledger_list)

Sharing list reflects changes: [99.99]


### 13. Shared References with Immutables
**Explanation**: Immutable objects (integers, floats, strings, tuples, frozensets) cannot be modified after creation. When modifying an immutable variable (e.g., `x += 1` or `s += '!'`), Python creates a brand new object in memory and re-binds the variable name to the new object, leaving any other references to the original object unchanged and thread-safe.

**Syntax**: `immutable_var += update_value  # Re-binds to new object`

In [14]:
original_limit_value = 500
sharing_limit_value = original_limit_value
original_limit_value = 600
print('Original limit modified, sharing limit stays same:', sharing_limit_value)

Original limit modified, sharing limit stays same: 500


### 14. Local Shadowing of Globals
**Explanation**: When a variable name inside a function matches a global variable name, the local definition shadows the global one within that function's scope. If you try to read a global variable and assign to it in the same function without `global`, Python's compiler treats it as local throughout the function, causing an `UnboundLocalError` on access prior to assignment.

**Syntax**: `def func(): global_var_name = local_value  # Shadows global`

In [15]:
global_currency_code = 'USD'
def set_local_currency():
    global_currency_code = 'EUR'
    print('Local currency inside function:', global_currency_code)
set_local_currency()
print('Global currency remains unchanged:', global_currency_code)

Local currency inside function: EUR
Global currency remains unchanged: USD


### 15. Dynamic Namespace Resolution (`locals()` & `globals()`)
**Explanation**: Python stores scope namespaces as dictionaries. `globals()` returns a direct dictionary reference to the module namespace, allowing dynamic variable inspection and mutation. `locals()` returns a dictionary representing the local scope; in function bodies, modifying `locals()` does not reliably update the actual local variables due to `LOAD_FAST` optimizations.

**Syntax**: `globals()['var_name'] = value` / `locals()`

In [16]:
global_namespace_dict = globals()
print('Is global_currency_code in globals?:', 'global_currency_code' in global_namespace_dict)

Is global_currency_code in globals?: True


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Practical 5+ years interview scenarios testing deep understanding of memory references, CPython object caching, and garbage collector tuning in high-throughput financial data pipelines.


In [17]:
# Solution:
import sys
with open(csv_path, 'r') as f:
    f.readline()
    row = f.readline().strip().split(',')
    card_provider_name = row[4]
    print('Card Provider:', card_provider_name, '| Refs count:', sys.getrefcount(card_provider_name) - 1)


Card Provider: Visa | Refs count: 3


### Q1: Evaluate Reference Counts & CPython Caching
**Explanation**: **Scenario**: Inspect the reference counts and object identity of string columns from the fintech dataset. Determine whether string interning or small integer caching applies to category codes.

**Syntax**: `sys.getrefcount(target)` / `target is cached_target`

In [18]:
# Solution:
import gc
first_cyclic_dict = {}
second_cyclic_dict = {}
first_cyclic_dict['link'] = second_cyclic_dict
second_cyclic_dict['link'] = first_cyclic_dict
del first_cyclic_dict, second_cyclic_dict
print('Cycles collected:', gc.collect())


Cycles collected: 2
